# **FICO® Xpress Solver Training**

***Xpress_solver_training.ipynb*** - FICO Xpress solver training exercises (Python)

&copy; Copyright 2026 Fair Isaac Corporation. The use of this example is subject to [legal and license requirements](https://github.com/fico-xpress/xpress-training#legal-and-license-requirements).

In [ ]:
# Install the necessary packages
%pip install -q xpress scipy matplotlib pandas

In [ ]:
import xpress as xp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter('ignore', xp.LicenseWarning)

## 📚 Course Overview

Welcome to this hands-on course on advanced solver usage with the **FICO® Xpress Python interface**.

This course is aimed at users who are already familiar with building optimization models in Xpress and want to deepen their understanding of the **solver** itself: how it works, how to tune it, and how to handle challenging situations.

| # | Topic | Slides |
|---|-------|--------|
| 1 | Solving and tuning a MIP problem | [MIP solving and tuning](../slides/MIPTheoryPypr.pdf) |
| 2 | Tolerances and scaling | [Tolerances and scaling](../slides/TolerScalPypr.pdf) |
| 3 | Infeasibility handling | [Infeasibility handling](../slides/InfeasPypr.pdf) |
| 4 | Nonlinear solving | [Nonlinear solving and tuning](../slides/NonLinearPypr.pdf) |
| 5 | Multiple MIP solutions | [Multiple MIP solutions](../slides/MIPSolsPypr.pdf) |
| 6 | Multi-objective optimization | [Multi-objective optimization](../slides/MultiObjPypr.pdf) |

## 💻 Setup Instructions

##### **Option 1: [GitHub Codespaces](https://github.com/features/codespaces) (Recommended)** ☁️
- Run this notebook in the cloud - no installation needed!
- You only need a [GitHub](https://github.com/signup) account.
- Upload the notebook to a codespace created from our [python-notebooks](https://github.com/fico-xpress/python-notebooks?tab=readme-ov-file) public repo, or use [Google Colab](https://colab.research.google.com/).

##### **Option 2: Local Setup** 🖥️
- Python >= 3.10 and <= 3.14
- Jupyter Notebook or a compatible IDE (VS Code, PyCharm)
- Install packages: `pip install xpress scipy matplotlib pandas`

## 📖 How to Use This Notebook

Two versions are provided:
- **`Xpress_solver_training.ipynb`** (this file) - exercises with `# TODO` items for you to complete
- **`Xpress_solver_training_Solutions.ipynb`** - same notebook with all solutions filled in

Work through the exercises in order. Code cells marked **🎯 TODO** require you to write or modify code before running them.

## 🔗 Documentation and Resources

- [Xpress Solver reference manual](https://www.fico.com/fico-xpress-optimization/docs/latest/solver)
- [Python Interface Reference Manual](https://www.fico.com/fico-xpress-optimization/docs/latest/solver/optimizer/python/HTML/GUID-616C323F-05D8-3460-B0D7-80F77DA7D046.html)
- [Xpress Python Examples](https://www.fico.com/fico-xpress-optimization/docs/latest/solver/optimizer/python/HTML/chExamples.html)
- [Controls reference](https://www.fico.com/fico-xpress-optimization/docs/latest/solver/optimizer/HTML/GUID-ACD4AC71-FE7B-3E80-A0AB-8A2B9B5E4E7A.html)

## 🔗 Related Public Notebooks

The [fico-xpress/python-notebooks](https://github.com/fico-xpress/python-notebooks) repository contains standalone examples related to topics in this course:

| Exercise | Related notebooks |
|----------|------------------|
| 3 - Infeasibility | [diagnose_infeasible.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/diagnose_infeasible.ipynb), [portfolio_pandas.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/portfolio_pandas.ipynb) |
| 4.1 - Local vs Global NLP | [circle_packing.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/circle_packing.ipynb), [inscribed_square.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/inscribed_square.ipynb) |
| 4.2 - PWL | [piecewise_linear.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/basic_api_examples/piecewise_linear.ipynb) |
| 5 - MIP callbacks | [tsp_callbacks.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/tsp_callbacks.ipynb), [callback_newnode.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/basic_api_examples/callback_newnode.ipynb) |
| 6 - Multi-objective | [markowitz_multiobj.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/markowitz_multiobj.ipynb), [multiobj_knapsack.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/basic_api_examples/multiobj_knapsack.ipynb) |

## Exercise 1 - Solving and tuning a MIP problem

*Slides: [MIP solving and tuning](../slides/MIPTheoryPypr.pdf)*

We explore how `heuremphasis` and `cutstrategy` affect MIP solver performance.
The test problem is an instance from the [MIPLIB 2017 collection](https://miplib.zib.de/tag_collection.html).

### 1.1. Heuristic emphasis

The [`heuremphasis`](https://www.fico.com/fico-xpress-optimization/docs/latest/solver/optimizer/HTML/HEUREMPHASIS.html) control specifies emphasis for primal heuristics.

| Value | Meaning |
|-------|---------|
| -1 | Auto (default) |
| 0 | Disable heuristics |
| 1 | Standard heuristics |
| 2 | Strong heuristics (finds more integer solutions early) |

**Try**: run the cell with `heuremphasis = 0`, then change it to `1` and `2`. Observe how the number of MIP solutions found changes in the solver log.

In [ ]:
import xpress as xp

p = xp.problem()
p.readProb('m_10teams.mps')

# TODO: Change heuremphasis to 0, 1, and 2 and re-run to compare solver behaviour
p.controls.heuremphasis = 0
p.controls.outputlog = 1

p.optimize()
print(f'Best bound: {p.attributes.bestbound:.4f}')
print(f'MIP solutions found: {p.attributes.mipsols}')

### 1.2. Cut strategy

The [`cutstrategy`](https://www.fico.com/fico-xpress-optimization/docs/latest/solver/optimizer/HTML/CUTSTRATEGY.html) control determines how aggressively Xpress generates cutting planes.

| Value | Meaning |
|-------|---------|
| -1 | Auto (default) |
| 0 | No cuts |
| 1 | Conservative |
| 2 | Moderate |
| 3 | Aggressive |

**Try**: run with `cutstrategy = 0` (no cuts) vs `3` (aggressive). Observe the impact on the LP relaxation bound and total solve time.

In [ ]:
p = xp.problem()
p.readProb('m_10teams.mps')

p.controls.heuremphasis = -1   # restore default
# TODO: Change cutstrategy to 0 (no cuts) then 3 (aggressive) and compare
p.controls.cutstrategy = 0
p.controls.outputlog = 1

p.optimize()
print(f'Best bound: {p.attributes.bestbound:.4f}')
print(f'Nodes explored: {p.attributes.nodes}')

## Exercise 2 - Tolerances and scaling

*Slides: [Tolerances and scaling](../slides/TolerScalPypr.pdf)*

Numerical tolerances determine when Xpress considers a solution feasible or a variable integral.
In poorly-scaled problems, the default tolerances may give incorrect results.

### 2.1. Feasibility tolerance (LP)

The feasibility tolerance (`feastol`, default `1e-6`) determines when a constraint violation is acceptable.
In poorly-scaled problems this can cause Xpress to report a feasible solution to an infeasible problem.

In [ ]:
import xpress as xp

# This problem is infeasible: x1 - x2 = 1.0000001 AND x1 <= 1 forces x2 < 0,
# but the constraint x1 - x2 = 1.0000001 conflicts with x2 >= 0.
# With the default feasibility tolerance, Xpress may accept this as feasible.

p = xp.problem()
p.controls.outputlog = 0

x1 = p.addVariable()
x2 = p.addVariable()

p.addConstraint(x1 - x2 == 1.0000001)
p.addConstraint(x1 <= 1)
p.setObjective(x1, sense=xp.maximize)

p.optimize()
print(f'Solution status: {p.attributes.solstatus}')
print(f'Solution: {p.getSolution()}')

🎯 **TODO 2.1**: The problem above reports a (wrong) feasible solution.
Decrease `feastol` to a value between `1e-9` and `1e-7` so that Xpress detects the true infeasibility.

In [ ]:
# TODO: Set feastol to a tighter value (between 1e-9 and 1e-7) and re-solve
p.controls.feastol = 1e-6  # <-- change this value

p.optimize()
print(f'Solution status: {p.attributes.solstatus}')

In [ ]:
import xpress as xp

# Scaling problem: coefficients span 10^9, causing the default tolerance to report
# a wrong feasible solution for what is actually an infeasible problem.

p = xp.problem()
p.controls.outputlog = 0
p.controls.presolve = 0
p.controls.scaling = 0  # disable auto-scaling to expose the numerical issue

x1 = p.addVariable()
x2 = p.addVariable(lb=-xp.infinity)
x3 = p.addVariable()

p.addConstraint((10**9)*x1 + x2 - x3 == 0)
p.addConstraint(x2 >= 100)
p.addConstraint(x1 == x3)

p.setObjective(x1 + x3, sense=xp.maximize)

p.optimize()
print(f'Solution status: {p.attributes.solstatus}')
print(f'Solution: {p.getSolution()}')

In [ ]:
# TODO: Set feastol to 1e-7 and re-solve to detect the true infeasibility
p.controls.feastol = 1e-6  # <-- change this value

p.optimize()
print(f'Solution status: {p.attributes.solstatus}')

### 2.2. MIP tolerance

The MIP integrality tolerance (`miptol`, default `1e-5`) determines the range within which a variable
value is considered integer. For problems with very large coefficients this may cause incorrect rounding.

In [ ]:
import xpress as xp

# With large coefficients, y may appear integral when it is actually 0 (not 1).
p = xp.problem()
p.controls.outputlog = 0
p.controls.presolve = 0

x = p.addVariable()
y = p.addVariable(vartype=xp.integer)

p.addConstraint(x <= 1000000 * y)
p.addConstraint(x >= 1000001)
p.setObjective(y)

p.optimize()
print(f'Solution: x={p.getSolution(x):.0f}, y={p.getSolution(y):.6f} (expected y=1)')

🎯 **TODO 2.2**: The solution above has `y` close to 0 instead of 1.
Tighten both `miptol` and `feastol` to `1e-7` to get the correct result.

In [ ]:
# TODO: Set miptol and feastol to 1e-7 to get the correct integer solution
p.controls.miptol = 1e-5   # <-- tighten this
p.controls.feastol = 1e-6  # <-- and this

p.optimize()
print(f'Solution: x={p.getSolution(x):.0f}, y={p.getSolution(y):.6f}')

## Exercise 3 - Infeasibility handling

*Slides: [Infeasibility handling](../slides/InfeasPypr.pdf)*

When a problem is infeasible, Xpress can compute an **Irreducible Infeasible Subset (IIS)**: the smallest
subset of constraints that are collectively infeasible. We will use a reusable helper function based on
the pattern in the public notebook [diagnose_infeasible.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/diagnose_infeasible.ipynb).
See also [portfolio_pandas.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/portfolio_pandas.ipynb) for the portfolio model used as the example here.

### Helper function: `diagnose_infeasible()`

The function below wraps the Xpress IIS API into a convenient interface.
**Feel free to copy this function and use it in your own projects.**

In [ ]:
import xpress as xp

def diagnose_infeasible(prob: xp.problem, find_isolations: bool = True, find_all: bool = False) -> dict:
    """
    Diagnose an infeasible problem by generating and analyzing IIS.

    Parameters:
        prob: An Xpress problem that has been solved and found infeasible.
        find_isolations: Whether to identify IIS isolations (constraints whose
                        removal would resolve the infeasibility without worsening
                        other independent IIS).
        find_all: If True, find all independent IIS. If False (default), only the first.

    Returns:
        dict with keys: is_infeasible, iis_found, num_iis, iis_list, message.
        iis_list is a list of dicts, one per IIS, with iis_constraints, iis_variables,
        isolation_constraints, isolation_variables, constraint_types, bound_types.
    """
    result = {'is_infeasible': False, 'iis_found': False, 'iis_status': 0,
              'num_iis': 0, 'iis_list': [], 'message': ''}

    if prob.attributes.solstatus != xp.SolStatus.INFEASIBLE:
        result['message'] = f"Problem is not infeasible. Status: {prob.attributes.solstatus.name}"
        return result

    result['is_infeasible'] = True
    if find_all:
        prob.IISAll()
    else:
        prob.firstIIS(1)

    iis_sol_status = prob.attributes.iissolstatus
    result['iis_status'] = iis_sol_status

    if iis_sol_status == xp.IISSolStatus.UNSTARTED:
        result['message'] = "IIS computation failed (license or input error)."
        return result
    if iis_sol_status == xp.IISSolStatus.FEASIBLE:
        result['message'] = "No IIS found - problem may be feasible."
        return result

    result['iis_found'] = True
    numiis, _, _, _, _ = prob.IISStatus()
    result['num_iis'] = numiis
    lines = ["INFEASIBILITY DIAGNOSIS", "=" * 50]
    if iis_sol_status == xp.IISSolStatus.UNFINISHED:
        lines.append("WARNING: IIS computation interrupted - subsystem may not be minimal")

    for iis_num in range(1, numiis + 1):
        if find_isolations:
            prob.IISIsolations(iis_num)
        miisrow, miiscol, constrainttype, colbndtype, duals, rdcs,             isolationrows, isolationcols = prob.getIISData(iis_num)

        constraints = prob.getConstraint(miisrow)
        variables   = prob.getVariable(miiscol)
        iis_info = {
            'iis_number': iis_num,
            'iis_constraints': constraints,
            'iis_variables': variables,
            'constraint_types': list(constrainttype),
            'bound_types': list(colbndtype),
            'isolation_constraints': [constraints[i] for i in range(len(miisrow)) if isolationrows[i] == 1],
            'isolation_variables':   [variables[i]   for i in range(len(miiscol)) if isolationcols[i] == 1],
        }
        result['iis_list'].append(iis_info)

        lines.append(f"\nIIS {iis_num}: {len(miisrow)} rows, {len(miiscol)} bounds")
        if miisrow:
            con_names = [c.name for c in constraints]
            nw = max(22, max(len(n) for n in con_names) + 2)
            lines.append(f"{'Row':<8} {'Name':<{nw}} {'Type':<8} {'Isolation'}")
            lines.append("-" * (8 + nw + 20))
            for i in range(len(miisrow)):
                iso = "YES" if isolationrows[i] == 1 else ""
                lines.append(f"{miisrow[i]:<8} {con_names[i]:<{nw}} {constrainttype[i]:<8} {iso}")
        if miiscol:
            var_names = [v.name for v in variables]
            vw = max(22, max(len(n) for n in var_names) + 2)
            lines.append(f"\n{'Col':<8} {'Variable':<{vw}} {'BndType':<10} {'Isolation'}")
            lines.append("-" * (8 + vw + 22))
            for i in range(len(miiscol)):
                iso = "YES" if isolationcols[i] == 1 else ""
                lines.append(f"{miiscol[i]:<8} {var_names[i]:<{vw}} {colbndtype[i]:<10} {iso}")

    if numiis > 1:
        lines.append(f"\nNote: {numiis} independent IIS. All must be resolved for feasibility.")

    result['message'] = "\n".join(lines)
    return result

### Portfolio optimization - infeasibility example

The following portfolio model has conflicting constraints. We will use `diagnose_infeasible()` to identify them.

In [ ]:
import xpress as xp
import pandas as pd

MaxHighRisk = 1/3
MinRegion = 0.1
MaxRegion = 0.2
MaxSector = 0.1
MinPerShare = 0.1
MaxPerShare = 0.2
MaxShares = 10

Shares = pd.read_csv("folio5.csv", index_col=['Shares'])
REGIONS = Shares['Location'].dropna().unique().tolist()
SECTORS = Shares['Sector'].unique().tolist()

p = xp.problem("portfolio")
p.controls.outputlog = 0

Shares['fraction'] = p.addVariables(Shares.index, vartype=xp.continuous, name='fraction')
Shares['buy']      = p.addVariables(Shares.index, vartype=xp.binary,     name='buy')

p.setObjective(xp.Sum(Shares.Return * Shares.fraction), sense=xp.maximize)

p.addConstraint(xp.Sum(Shares.loc[Shares.Risk, 'fraction']) <= MaxHighRisk)
p.addConstraint([xp.Sum(Shares.loc[Shares['Location'] == r, 'fraction']) >= MinRegion for r in REGIONS])
p.addConstraint([xp.Sum(Shares.loc[Shares['Location'] == r, 'fraction']) <= MaxRegion for r in REGIONS])
p.addConstraint([xp.Sum(Shares.loc[Shares['Sector'] == t,   'fraction']) <= MaxSector for t in SECTORS])
p.addConstraint(xp.Sum(Shares.fraction) == 1, name="budget")
p.addConstraint(Shares.fraction[i] <= MaxPerShare for i in Shares.index)
p.addConstraint(xp.Sum(Shares.buy) <= MaxShares, name="max_holdings")
p.addConstraint((Shares.fraction[i] >= MinPerShare * Shares.buy[i] for i in Shares.index), name="min_fraction")
p.addConstraint((Shares.fraction[i] <= MaxPerShare * Shares.buy[i] for i in Shares.index), name="max_fraction")

p.optimize()
print(f'Solution status: {p.attributes.solstatus}')

🎯 **TODO 3.1**: The problem is infeasible. Call `diagnose_infeasible(p)` and print its `message` field
to identify which constraints form the IIS.

In [ ]:
# TODO 3.1: Call diagnose_infeasible(p) and print result['message']
result = None  # replace with diagnose_infeasible(p)
# print(result['message'])

🎯 **TODO 3.2**: Based on the IIS output, the sector diversification constraints are too tight.
For each constraint identified as an **isolation** with type `'L'` (less-than-or-equal), relax its RHS to `0.2`
until the problem becomes feasible.

In [ ]:
# TODO 3.2: For each IIS isolation row of type 'L', increase its RHS to 0.2
# Hint: use p.chgRHS([row_index], [new_value]) and re-optimize after each change
# Loop until p.attributes.solstatus != xp.SolStatus.INFEASIBLE

for iis_info in result['iis_list']:
    pass  # replace with your fix logic

### Infeasibility repair utility (optional)

As an alternative to manual diagnosis, `repairInfeas()` automatically relaxes constraints to restore feasibility.

In [ ]:
# Re-build the infeasible model (repair modifies in-place)
p.optimize()  # re-solve original (infeasible) state

penalty = 'c'
phase2  = 'o'
flags   = 'g'
delta   = 0
lepref, gepref, lbpref, ubpref = 1, 0, 0, 0

p.repairInfeas(penalty, phase2, flags, lepref, gepref, lbpref, ubpref, delta)

print(f'Repaired objective: {p.attributes.objval:.4f}')
alloc = [(s, round(p.getSolution(v), 3)) for s, v in zip(Shares.index, Shares["fraction"])]
print(f'Allocation (top 5): {alloc[:5]}')

## Exercise 4 - Nonlinear solving

*Slides: [Nonlinear solving and tuning](../slides/NonLinearPypr.pdf)*

### 4.1. Local vs Global NLP solver

**Problem**: Pack $N$ circles inside a unit square to **maximize the sum of their radii**.

This is a nonconvex NLP. A **local solver** (SLP) is fast but may stop at a local optimum.
A **global solver** searches exhaustively and guarantees the global optimum.

*Related public notebooks: [circle_packing.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/circle_packing.ipynb), [inscribed_square.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/inscribed_square.ipynb)*

In [ ]:
import xpress as xp
import matplotlib.pyplot as plt
import numpy as np

N = 6

p = xp.problem("Circle_Packing")
p.controls.outputlog = 0

x = p.addVariables(N, name="x")
y = p.addVariables(N, name="y")
r = p.addVariables(N, name="r", ub=0.5)

p.addConstraint(
    (x[i] - x[j])**2 + (y[i] - y[j])**2 >= (r[i] + r[j])**2
    for i in range(N) for j in range(i + 1, N)
)
p.addConstraint(x[i] >= r[i] for i in range(N))
p.addConstraint(x[i] <= 1 - r[i] for i in range(N))
p.addConstraint(y[i] >= r[i] for i in range(N))
p.addConstraint(y[i] <= 1 - r[i] for i in range(N))

p.setObjective(xp.Sum(r), sense=xp.maximize)
print(f"Model: {N} circles, {p.attributes.cols} variables, {p.attributes.rows} constraints")

🎯 **TODO 4.1**: Solve the circle packing model with the **local SLP solver** and record the objective value.

In [ ]:
# TODO 4.1: Set nlpsolver to NLPSOLVER_LOCAL and localsolver to LOCALSOLVER_XSLP, then optimize
p.controls.nlpsolver   = xp.constants.NLPSOLVER_LOCAL
p.controls.localsolver = xp.constants.LOCALSOLVER_XSLP  # SLP

# TODO: call p.optimize() and print sum of radii

🎯 **TODO 4.2**: Now switch to the **global solver** and compare. The global solver should find a better solution.

In [ ]:
# TODO 4.2: Change nlpsolver to NLPSOLVER_GLOBAL, set a time limit, and re-solve
p.controls.nlpsolver  = xp.constants.NLPSOLVER_GLOBAL
p.controls.timelimit  = 10

# TODO: call p.optimize() and compare the result to the local solver

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, x_sol, y_sol, r_sol, title in [
    (ax1, x_local,  y_local,  r_local,  f'Local  (sum={sum(r_local):.4f})'),
    (ax2, x_global, y_global, r_global, f'Global (sum={sum(r_global):.4f})'),
]:
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
    ax.grid(True, alpha=0.3); ax.set_title(title, fontweight='bold')
    for i in range(N):
        ax.add_patch(plt.Circle((x_sol[i], y_sol[i]), r_sol[i],
                                edgecolor='steelblue', facecolor='lightblue', alpha=0.6, linewidth=2))
        ax.text(x_sol[i], y_sol[i], f'{r_sol[i]:.2f}', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()

### 4.2. Using PWL to approximate a nonlinear function

We approximate $\sin(freq \cdot x)$ over $[0, 2/\pi]$ using $N$ piecewise-linear breakpoints.
Increase $N$ to improve accuracy.

*Related public notebook: [piecewise_linear.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/basic_api_examples/piecewise_linear.ipynb)*

🎯 **TODO 4.3**: Run the cell with `N = 5`, then try `N = 10, 15, 20` and observe the improvement.

In [ ]:
import xpress as xp
import math
import matplotlib.pyplot as plt

def create_segments(N, freq):
    step = (2 / math.pi) / (N - 1)
    breakpoints = xp.array([i * step for i in range(N)])
    values = xp.sin(freq * breakpoints)
    return breakpoints, values

p = xp.problem()
p.controls.outputlog = 0

x = p.addVariable(lb=0, ub=2/math.pi)
y = p.addVariable(lb=-1, ub=1)

N    = 10       # TODO: try 5, 10, 15, 20
freq = 5

breakpoints, values = create_segments(N, freq)

# Build piecewise-linear function: y = sin(freq * x) approximated by N-1 segments
slopes = [(values[i+1] - values[i]) / (breakpoints[i+1] - breakpoints[i]) for i in range(N-1)]
pw = xp.pwl({(breakpoints[i], breakpoints[i+1]):
    values[i] + slopes[i] * (x - breakpoints[i]) for i in range(N-1)})
p.addConstraint(y == pw)
p.setObjective(y, sense=xp.maximize)

p.optimize()

# Visualize the approximation
x_exact  = np.linspace(0, 2/math.pi, 300)
y_exact  = np.sin(freq * x_exact)
x_breaks = breakpoints.tolist()
y_breaks = values.tolist()

plt.figure(figsize=(9, 4))
plt.plot(x_exact, y_exact, label='Exact sin(freq*x)', linewidth=2)
plt.plot(x_breaks, y_breaks, 'o--', label=f'PWL approx (N={N})', linewidth=1.5, markersize=5)
plt.axvline(p.getSolution(x), color='red', linestyle=':', label=f'Optimal x={p.getSolution(x):.3f}')
plt.legend(); plt.grid(True, alpha=0.3)
plt.title(f'PWL approximation of sin({freq}x) with N={N} breakpoints')
plt.tight_layout(); plt.show()
print(f"Optimal: x={p.getSolution(x):.4f}, y={p.getSolution(y):.4f}")

## Exercise 5 - Multiple MIP solutions

*Slides: [Multiple MIP solutions](../slides/MIPSolsPypr.pdf)*

Xpress stores MIP solutions as it finds them. You can also use **callbacks** to inspect or export
each integer solution as it is discovered during branch and bound.

*Related public notebooks: [tsp_callbacks.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/tsp_callbacks.ipynb), [callback_newnode.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/basic_api_examples/callback_newnode.ipynb)*

### 5.1. Storing multiple MIP solutions via callbacks

**Problem**: choose a pop group from 16 candidates to maximize a stardom score,
subject to constraints on studio slots, vocal lessons, PR minders, and cost.

In [ ]:
import xpress as xp
import numpy as np
import pandas as pd

STARS   = ['Martin','Caroline','Stewart','Richard','Claire','Lynn','Hilary','Suzanne',
           'David','Michael','Phil','Tom','Rachael','Julia','Emma','Matthew']
STARDOM = np.array([0.4386,0.397,0.5364,0.3659,0.3923,0.6155,0.6579,0.6155,
                    0.4737,0.2416,0.4347,0.442,0.4576,0.5309,0.4178,0.3857])
STUDIO  = np.array([1,2,3,0,1,4,0,4,1,0,1,0,1,0,1,2])
VOCAL   = np.array([2,1,6,0,1,2,0,4,0,0,0,0,0,1,2,0])
PR      = np.array([3,1,4,1,0,2,1,4,3,0,1,0,0,0,0,0])
COST    = np.array([5,10,6,10,8,12,7,8,6,5,9,10,7,11,8,9])

data = {'STARS': STARS, 'STARDOM': pd.Series(STARDOM),
        'STUDIO': pd.Series(STUDIO), 'VOCAL': pd.Series(VOCAL),
        'PR': pd.Series(PR), 'COST': pd.Series(COST)}

p = xp.problem("pop_group")
p.controls.outputlog = 0

select = p.addVariables(len(STARS), vartype=xp.binary, name='select')
studio_spare = p.addVariable(name='studio_spare')
vocal_spare  = p.addVariable(name='vocal_spare')

p.addConstraint(xp.Sum(STUDIO * select) + studio_spare == 12)
p.addConstraint(xp.Sum(VOCAL  * select) + vocal_spare  == 14)
p.addConstraint(xp.Sum(PR     * select) <= 25)
p.addConstraint(xp.Sum(COST   * select) <= 50)
p.addConstraint(3 <= xp.Sum(select))
p.addConstraint(xp.Sum(select) <= 8)

p.setObjective(xp.Sum(STARDOM * select), sense=xp.maximize)
print(f"Model created: {len(STARS)} candidates, {p.attributes.cols} variables")

🎯 **TODO 5.1**: Define a callback function `print_solution(prob, data)` that is called each time
a new integer solution is found. The callback should print the stardom score and the selected members.
Then register it with `p.addIntsolCallback(print_solution, data)` and call `p.optimize()`.

In [ ]:
# TODO 5.1: Define a callback function and register it
def print_solution(prob, data):
    sol = np.array(prob.getCallbackSolution())
    # TODO: print the solution number, stardom score, and selected band members
    pass

# TODO: Register the callback and optimize
# p.addIntsolCallback(print_solution, data)
# p.optimize()

### Using `mipaddcutoff` to accept near-optimal solutions

`mipaddcutoff` offsets the cutoff threshold: a negative value makes Xpress accept solutions
that are slightly worse than the current best, producing a richer set of near-optimal alternatives.

In [ ]:
p2 = xp.problem("pop_group_cutoff")
p2.controls.outputlog = 0

# Rebuild same model
select2 = p2.addVariables(len(STARS), vartype=xp.binary, name='select')
p2.addConstraint(xp.Sum(STUDIO * select2) <= 12)
p2.addConstraint(xp.Sum(VOCAL  * select2) <= 14)
p2.addConstraint(xp.Sum(PR     * select2) <= 25)
p2.addConstraint(xp.Sum(COST   * select2) <= 50)
p2.addConstraint(3 <= xp.Sum(select2))
p2.addConstraint(xp.Sum(select2) <= 8)
p2.setObjective(xp.Sum(STARDOM * select2), sense=xp.maximize)

p2.controls.mipaddcutoff = -0.5  # accept solutions up to 0.5 units below the best found
p2.addIntsolCallback(print_solution, data)
p2.optimize()
print(f"\nTotal solutions recorded: {p2.attributes.mipsols}")

### 5.2. Adding MIP solutions

You can also inject a heuristic solution (feasible or partial) into Xpress using `addMipSol()`.
This can warm-start the solver with a good initial point.

In [ ]:
p3 = xp.problem("pop_group_warmstart")
p3.controls.outputlog = 0

select3 = p3.addVariables(len(STARS), vartype=xp.binary, name='select')
p3.addConstraint(xp.Sum(STUDIO * select3) <= 12)
p3.addConstraint(xp.Sum(VOCAL  * select3) <= 14)
p3.addConstraint(xp.Sum(PR     * select3) <= 25)
p3.addConstraint(xp.Sum(COST   * select3) <= 50)
p3.addConstraint(3 <= xp.Sum(select3))
p3.addConstraint(xp.Sum(select3) <= 8)
p3.setObjective(xp.Sum(STARDOM * select3), sense=xp.maximize)

# Inject 5 random feasible starting solutions
np.random.seed(42)
for k in range(5):
    sol = np.random.choice([0, 1], size=len(STARS))
    p3.addMipSol(solval=sol, name=f"heuristic_{k}")

p3.optimize()
print(f"Optimal: {p3.attributes.objval:.4f}, found in {p3.attributes.nodes} nodes")

## Exercise 6 - Multi-objective optimization

*Slides: [Multi-objective optimization](../slides/MultiObjPypr.pdf)*

Xpress supports multi-objective problems natively via the `setObjective(objidx=...)` API.
We demonstrate two approaches: **blended (weighted sum)** and **lexicographic**.

*Related public notebooks: [markowitz_multiobj.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/modeling_examples/markowitz_multiobj.ipynb), [multiobj_knapsack.ipynb](https://github.com/fico-xpress/python-notebooks/blob/main/basic_api_examples/multiobj_knapsack.ipynb)*

### 7.1. Efficient frontier - Markowitz portfolio

**Problem**: invest in 5 stocks to maximize expected return while minimizing variance.
These two objectives are in conflict: we trace the **Pareto-efficient frontier** by
solving weighted-sum problems across a range of weights.

In [ ]:
import xpress as xp
import numpy as np
import matplotlib.pyplot as plt

returns    = np.array([0.31, 0.87, 0.31, 0.66, 0.24])
covariance = np.array([
    [ 0.32,  0.70,  0.19,  0.52,  0.16],
    [ 0.70,  4.35, -0.48, -0.06, -0.03],
    [ 0.19, -0.48,  0.98,  1.10,  0.10],
    [ 0.52, -0.06,  1.10,  2.48,  0.37],
    [ 0.16, -0.03,  0.10,  0.37,  0.31]
])

p = xp.problem()
p.controls.outputlog = 0

x        = p.addVariables(len(returns))
variance = p.addVariable(lb=-xp.infinity)

p.addConstraint(xp.Sum(x) == 1)
p.addConstraint(xp.Dot(xp.Dot(covariance, x), x) <= variance)

# Objective 0: maximize return; Objective 1: minimize variance (use weight=-1 same sense in 9.9)
p.setObjective(xp.Dot(returns, x), objidx=0, sense=xp.maximize, weight=1)
p.setObjective(variance,           objidx=1, sense=xp.maximize, weight=-1)  # negative weight = minimize
print("Model created: 5-stock Markowitz portfolio")

🎯 **TODO 6.1**: Compute the efficient frontier by solving 20 blended problems with weights
$w = 0, 0.05, 0.10, \ldots, 1.0$ for the return objective (weight $1-w$ for variance).
Plot mean return vs variance for each solution.

In [ ]:
# TODO 6.1: Trace the efficient frontier
means     = []
variances = []

for w in np.linspace(0, 1, 20):
    # TODO: set weights for both objectives (same sense=maximize, use negative weight for variance)
    # p.setObjective(objidx=0, weight=w, sense=xp.maximize)
    # p.setObjective(objidx=1, weight=-(1-w), sense=xp.maximize)
    # p.optimize()
    # then collect means and variances
    pass

plt.figure(figsize=(8, 5))
plt.plot(means, variances, 'b-o', markersize=5)
plt.xlabel('Expected return'); plt.ylabel('Variance')
plt.title('Efficient frontier'); plt.grid(True, alpha=0.3); plt.show()

### 7.2. Lexicographic approach

In the lexicographic approach, objectives are prioritized: optimize the first objective to
optimality, then optimize the second within the tolerance set by `reltol` on the first.

In [ ]:
p.setObjective(objidx=0, priority=1, weight=1, reltol=0.1, sense=xp.maximize)
p.setObjective(objidx=1, priority=0, weight=-1)
p.controls.outputlog = 0
p.optimize()

m0 = xp.Dot(p.getSolution(x), returns).item()
v0 = p.getSolution(variance)
print(f"Lexicographic solution: return={m0:.4f}, variance={v0:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(variances, means, 'b-o', markersize=5, label='Efficient frontier')
plt.plot(v0, m0, 'r*', markersize=15, label='Lexicographic solution')
plt.xlabel('Variance'); plt.ylabel('Expected return')
plt.title('Efficient frontier with lexicographic solution')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

### 7.3. Goal programming (optional)

An example of lexicographic goal programming: balance production of two products
against department capacity goals using deviational variables.

In [ ]:
import xpress as xp

p2 = xp.problem()
p2.controls.outputlog = 0

produceA = p2.addVariable(vartype=xp.integer)
produceB = p2.addVariable(vartype=xp.integer)

surplus_wiring   = p2.addVariable()
deficit_wiring   = p2.addVariable()
surplus_assembly = p2.addVariable()
deficit_assembly = p2.addVariable()
deficit_profit   = p2.addVariable()
deficit_productB = p2.addVariable()

p2.addConstraint(2*produceA + 4*produceB + deficit_profit - surplus_wiring == 160,   name="profit_goal")
p2.addConstraint(3*produceA + 2*produceB + deficit_wiring - surplus_wiring == 120,    name="wiring_goal")
p2.addConstraint(2*produceA + 3*produceB + deficit_assembly - surplus_assembly == 100, name="assembly_goal")
p2.addConstraint(produceB + deficit_productB == 40,                                    name="productB_goal")

# Lexicographic goal: minimize profit deficit first (priority 3), then wiring deviation (priority 2), etc.
p2.setObjective(deficit_profit,                   objidx=0, priority=3, weight=1)
p2.setObjective(deficit_wiring + surplus_wiring,   objidx=1, priority=2, weight=1)
p2.setObjective(deficit_assembly + surplus_assembly, objidx=2, priority=1, weight=1)
p2.setObjective(deficit_productB,                  objidx=3, priority=0, weight=1)

p2.optimize()
print(f"Produce A: {p2.getSolution(produceA):.0f}, Produce B: {p2.getSolution(produceB):.0f}")
print(f"Profit deficit: {p2.getSolution(deficit_profit):.0f}")